
# SFTP to Lakehouse Mirror (Hybrid for Large CSVs)

This Fabric-ready notebook recursively mirrors an SFTP folder tree into your Lakehouse Files area while preserving subfolder structure and copying only `.csv` files (case-insensitive).

Hybrid strategy:
- Small files (below threshold): buffered write
- Large files (at or above threshold, default 1 GB): streaming write in chunks

Steps:
1. Edit the CONFIG cell with your SFTP details.
2. Optionally start with dry_run = True.
3. Run all cells top to bottom.


In [ ]:
%pip install -q paramiko

In [ ]:

import io, os, stat, time, traceback
from typing import List, Optional

try:
    from notebookutils import mssparkutils
except ImportError:
    import mssparkutils

import paramiko


def join_path(*parts: str) -> str:
    return "/".join([p.strip("/").replace("\", "/") for p in parts if p]).replace("//", "/")


def ensure_dir(path: str) -> None:
    if not mssparkutils.fs.exists(path):
        mssparkutils.fs.mkdirs(path)


def is_dir(attr) -> bool:
    return stat.S_ISDIR(attr.st_mode)


def connect_sftp(host, port, username, password=None, key_path=None, key_passphrase=None):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    if password:
        ssh.connect(host, port=port, username=username, password=password,
                    look_for_keys=False, allow_agent=False, timeout=30)
    elif key_path:
        pkey = paramiko.RSAKey.from_private_key_file(key_path, password=key_passphrase)
        ssh.connect(host, port=port, username=username, pkey=pkey,
                    look_for_keys=False, allow_agent=False, timeout=30)
    else:
        raise RuntimeError("Password or key_path required")

    return ssh, ssh.open_sftp()


def copy_small(sftp, src, dst):
    ensure_dir(os.path.dirname(dst))
    with sftp.open(src, 'rb') as f:
        data = f.read()
    with mssparkutils.fs.open(dst, 'wb') as out:
        out.write(data)


def copy_large(sftp, src, dst, chunk_size):
    ensure_dir(os.path.dirname(dst))
    with sftp.open(src, 'rb') as f:
        with mssparkutils.fs.open(dst, 'wb') as out:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                out.write(chunk)


def mirror_sftp_to_lakehouse_hybrid(
    host: str,
    username: str,
    sftp_root: str,
    lakehouse_base: str,
    port: int = 22,
    password: Optional[str] = None,
    key_path: Optional[str] = None,
    key_passphrase: Optional[str] = None,
    include_exts: Optional[List[str]] = None,
    overwrite: bool = True,
    recursive: bool = True,
    dry_run: bool = False,
    large_file_threshold_mb: int = 1024,
    chunk_size_mb: int = 8,
):
    ssh, sftp = connect_sftp(host, port, username, password, key_path, key_passphrase)

    threshold = large_file_threshold_mb * 1024 * 1024
    chunk_size = chunk_size_mb * 1024 * 1024

    stack = [sftp_root]
    ensure_dir(lakehouse_base)

    try:
        while stack:
            cur = stack.pop()
            for e in sftp.listdir_attr(cur):
                src = join_path(cur, e.filename)
                if is_dir(e):
                    if recursive:
                        stack.append(src)
                else:
                    if include_exts and not any(src.lower().endswith(x) for x in include_exts):
                        continue

                    rel = src[len(sftp_root):].lstrip('/')
                    dst = join_path(lakehouse_base, rel)

                    if not overwrite and mssparkutils.fs.exists(dst):
                        continue

                    if dry_run:
                        print(f"Would copy {src} -> {dst}")
                        continue

                    size = sftp.stat(src).st_size
                    if size >= threshold:
                        copy_large(sftp, src, dst, chunk_size)
                    else:
                        copy_small(sftp, src, dst)
    finally:
        sftp.close()
        ssh.close()


In [ ]:

CONFIG = {
    "host": "sftp.example.com",
    "port": 22,
    "username": "myuser",
    "password": "mypassword",
    "key_path": None,
    "key_passphrase": None,
    "sftp_root": "/export/data",
    "lakehouse_base": "/lakehouse/default/Files/sftp_raw",
    "include_exts": [".csv"],
    "overwrite": True,
    "recursive": True,
    "dry_run": False,
    "large_file_threshold_mb": 1024,
    "chunk_size_mb": 8,
}

mirror_sftp_to_lakehouse_hybrid(**CONFIG)
